# 51 - Image-derived TimTrack peak stream into KLT and 2-state Kalman

This notebook removes the saved-MATLAB-geofeature oracle from the TimTrack/KLT handoff gate.

Flow:

1. Generate the full-sequence TimTrack peak stream from `UltraTimTrack_test.mp4` image frames.
2. Compare the Python peak stream against MATLAB saved full-sequence `Fdat.geofeatures` / `UTT_numeric_export` peaks.
3. Build KLT masks from the Python image-derived geofeatures.
4. Apply the one-step KLT handoff and feed that into the MATLAB-compatible 2-state Kalman gate.
5. Run `scripts/compare_ultratimtrack_parity.py` without the NB36 intermediate TimTrack export, so TimTrack rows target saved full-sequence MATLAB geofeatures.

The main parity CSV tests the current downstream gate by replacing only the saved geofeature source. A stricter image-TimTrack-prior variant is also saved and scored as a diagnostic.

In [7]:
from __future__ import annotations

from pathlib import Path
import json
import os
import subprocess
import sys
import time

os.environ.setdefault("MPLCONFIGDIR", "/private/tmp/matplotlib")

import cv2
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from scipy.io import loadmat

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from ultrasound_tracker.geometry import line_angles_batch, line_lengths_batch, normalize_angle
from ultrasound_tracker.matlab_compat import metric_row
from ultrasound_tracker.matlab_timtrack import (
    alpha_from_saved_peaks,
    extract_saved_peak_arrays,
    fascicle_segment_from_geofeature,
    reconstruct_saved_geofeature_alpha,
    run_timtrack_geofeatures_from_video,
    saved_alpha_error,
)
from ultrasound_tracker.ultratrack_klt import (
    UltraTrackKLTConfig,
    apply_affine_1b,
    run_one_step_affine_video,
)
from ultrasound_tracker.ultratimtrack_matlab_2state import (
    MatlabTwoStateKalmanConfig,
    run_matlab_2state_kalman,
)

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

OUT = ROOT / "results" / "notebook51_image_derived_timtrack_klt_pipeline_gate"
OUT.mkdir(parents=True, exist_ok=True)

MATLAB_EXPORT = Path("/Users/grosbedou/Documents/GitHub/UltraTimTrack/UTT_numeric_export.mat")
MATLAB_RESULT = ROOT / "data" / "matlab" / "slow_low_2.mat"
VIDEO_PATH = ROOT / "data" / "raw" / "UltraTimTrack_test.mp4"

IMAGE_TIMTRACK_NPZ = OUT / "image_derived_timtrack_geofeatures_arrays.npz"
KLT_NPZ = OUT / "python_geofeature_mask_one_step_klt_arrays.npz"
GATE_FINAL_NPZ = OUT / "image_timtrack_py_masks_matlab_klt_2state_final_arrays.npz"
STRICT_FINAL_NPZ = OUT / "strict_image_timtrack_prior_2state_final_arrays.npz"
PARITY_CSV = OUT / "parity_metrics.csv"

FORCE_REBUILD_TIMTRACK = os.environ.get("NB51_FORCE_REBUILD", "0") == "1"

for label, path in {
    "MATLAB numeric export": MATLAB_EXPORT,
    "MATLAB result": MATLAB_RESULT,
    "video": VIDEO_PATH,
}.items():
    print(f"{label:24s} {'OK' if path.exists() else 'MISSING'} {path}")
    if not path.exists():
        raise FileNotFoundError(path)
print("Output folder:", OUT)
print("Force TimTrack rebuild:", FORCE_REBUILD_TIMTRACK)

MATLAB numeric export    OK /Users/grosbedou/Documents/GitHub/UltraTimTrack/UTT_numeric_export.mat
MATLAB result            OK /Users/grosbedou/PycharmProjects/NDORMS/data/matlab/slow_low_2.mat
video                    OK /Users/grosbedou/PycharmProjects/NDORMS/data/raw/UltraTimTrack_test.mp4
Output folder: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook51_image_derived_timtrack_klt_pipeline_gate
Force TimTrack rebuild: False


In [8]:
mat_root = loadmat(MATLAB_EXPORT, simplify_cells=True)["UTT_numeric_export"]
region = mat_root["Region"]
fascicle = region["Fascicle"]
geofeatures = list(np.asarray(mat_root["geofeatures"], dtype=object).reshape(-1))
parms = mat_root["parms"]

height = int(mat_root["vidHeight"])
width = int(mat_root["vidWidth"])
mat_n = int(mat_root["NumFrames"])
mm_per_px = float(mat_root["ID"]) / height
block_size = np.asarray(mat_root["BlockSize"], dtype=int).reshape(-1)
win_size = (int(block_size[1]), int(block_size[0]))
apox_1b = np.asarray(parms["apo"]["apox"], dtype=float).reshape(-1)
max_peaks = int(parms["fas"].get("npeaks", 10))

cap = cv2.VideoCapture(str(VIDEO_PATH))
fps = float(cap.get(cv2.CAP_PROP_FPS)) if cap.isOpened() else np.nan
video_frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) if cap.isOpened() else -1
cap.release()


def matlab_pair_series(values: object, n: int | None = None) -> np.ndarray:
    out = []
    for value in np.asarray(values, dtype=object).reshape(-1):
        arr = np.asarray(value, dtype=float).reshape(-1)
        out.append(arr[:2] if arr.size >= 2 else [np.nan, np.nan])
    result = np.asarray(out, dtype=float)
    return result if n is None else result[:n]


def matlab_fascicle_segments(x_field: str, y_field: str, n: int | None = None) -> np.ndarray:
    x = matlab_pair_series(fascicle[x_field], n=n)
    y = matlab_pair_series(fascicle[y_field], n=n)
    return np.column_stack([x[:, 1], y[:, 1], x[:, 0], y[:, 0]])


def matlab_apo_segments(prefix: str, n: int | None = None) -> np.ndarray:
    x = matlab_pair_series(region[f"{prefix}_x"], n=n)
    y = matlab_pair_series(region[f"{prefix}_y"], n=n)
    return np.column_stack([x[:, 0], y[:, 0], x[:, 1], y[:, 1]])


def normalized_angles_deg(segments: np.ndarray) -> np.ndarray:
    return np.asarray([normalize_angle(v, degrees=True) for v in line_angles_batch(segments, degrees=True)], dtype=float)


mat_alpha_saved = np.asarray([float(entry["alpha"]) for entry in geofeatures], dtype=float)[:mat_n]
mat_alpha_rebuilt = reconstruct_saved_geofeature_alpha(geofeatures)[:mat_n]
saved_alpha, saved_peak_alphas, saved_peak_weights = extract_saved_peak_arrays(geofeatures, max_peaks=max_peaks)
saved_alpha = saved_alpha[:mat_n]
saved_peak_alphas = saved_peak_alphas[:mat_n]
saved_peak_weights = saved_peak_weights[:mat_n]

mat_raw_klt = matlab_fascicle_segments("fas_x_original", "fas_y_original", n=mat_n)
mat_sup = matlab_apo_segments("sup", n=mat_n)
mat_deep = matlab_apo_segments("deep", n=mat_n)
mat_final = {
    "FL_mm": np.asarray(region["fas_length"], dtype=float).reshape(-1)[:mat_n],
    "PEN_deg": np.asarray(region["fas_pen"], dtype=float).reshape(-1)[:mat_n],
    "ANG_deg": np.asarray(region["fas_ang"], dtype=float).reshape(-1)[:mat_n],
}

kalman_config = MatlabTwoStateKalmanConfig(
    q_parameter=float(mat_root["Q"]),
    x_measurement_variance=float(mat_root["X"]),
    alpha_measurement_variance=float(np.asarray(mat_root["R"], dtype=float).reshape(-1)[0]),
    n_start_frames=int(mat_root["NS"]),
    run_smoother=True,
)

print({
    "frames_matlab": mat_n,
    "frames_video": video_frame_count,
    "shape": (height, width),
    "fps": fps,
    "mm_per_px": mm_per_px,
    "apox_1b": apox_1b.tolist(),
    "max_peaks": max_peaks,
    "win_size": win_size,
    "saved_alpha_reconstruction": saved_alpha_error(geofeatures),
    "kalman_config": kalman_config,
})

{'frames_matlab': 2666, 'frames_video': 2667, 'shape': (562, 706), 'fps': 33.341, 'mm_per_px': 0.09021352313167261, 'apox_1b': [20.0, 94.0, 168.0, 242.0, 316.0, 390.0, 464.0, 538.0, 612.0, 686.0], 'max_peaks': 10, 'win_size': (71, 21), 'saved_alpha_reconstruction': {'n': 2666, 'max_abs_error_deg': 0.0, 'rmse_deg': 0.0}, 'kalman_config': MatlabTwoStateKalmanConfig(q_parameter=0.01, x_measurement_variance=100.0, alpha_measurement_variance=3.055292111054342, n_start_frames=1, run_smoother=True)}


In [9]:
def _scalar(entry: dict, key: str) -> float:
    value = np.asarray(entry.get(key, np.nan), dtype=float).reshape(-1)
    return float(value[0]) if value.size else np.nan


def _pad_vector(value, width: int) -> np.ndarray:
    arr = np.asarray(value, dtype=float).reshape(-1)
    out = np.full(int(width), np.nan, dtype=float)
    n = min(len(arr), int(width))
    if n:
        out[:n] = arr[:n]
    return out


def _pad_matrix(value, shape: tuple[int, int]) -> np.ndarray:
    arr = np.asarray(value, dtype=float)
    out = np.full(shape, np.nan, dtype=float)
    if arr.ndim == 2:
        r = min(arr.shape[0], shape[0])
        c = min(arr.shape[1], shape[1])
        out[:r, :c] = arr[:r, :c]
    return out


def _trim_nan_tail(row: np.ndarray) -> np.ndarray:
    arr = np.asarray(row, dtype=float).reshape(-1)
    valid = np.isfinite(arr)
    if not np.any(valid):
        return np.asarray([], dtype=float)
    return arr[: int(np.where(valid)[0][-1]) + 1]


def entries_to_timtrack_arrays(entries: list[dict]) -> dict[str, np.ndarray]:
    n = len(entries)
    coef_width = 4
    arrays: dict[str, np.ndarray] = {
        "frame": np.asarray([int(_scalar(entry, "frame")) for entry in entries], dtype=np.int32),
        "time_s": np.asarray([int(_scalar(entry, "frame")) / fps for entry in entries], dtype=np.float32),
        "success": np.asarray([np.isfinite(_scalar(entry, "alpha")) for entry in entries], dtype=bool),
        "ANG_deg": np.asarray([_scalar(entry, "alpha") for entry in entries], dtype=np.float32),
        "PEN_deg": np.asarray([_scalar(entry, "phi") for entry in entries], dtype=np.float32),
        "FL_px": np.asarray([_scalar(entry, "faslen") for entry in entries], dtype=np.float32),
        "fascicle_angle_deg": np.asarray([_scalar(entry, "alpha") for entry in entries], dtype=np.float32),
        "pennation_angle_deg": np.asarray([_scalar(entry, "phi") for entry in entries], dtype=np.float32),
        "fascicle_length_px": np.asarray([_scalar(entry, "faslen") for entry in entries], dtype=np.float32),
        "super_apo_angle_deg": np.asarray([_scalar(entry, "betha") for entry in entries], dtype=np.float32),
        "deep_apo_angle_deg": np.asarray([_scalar(entry, "gamma") for entry in entries], dtype=np.float32),
        "muscle_thickness_px": np.asarray([_scalar(entry, "thickness") for entry in entries], dtype=np.float32),
        "apo_x": np.asarray([_scalar(entry, "apo_x") for entry in entries], dtype=np.float32),
        "emask_vertical_radius": np.asarray([_pad_vector(entry.get("Emask_radius", []), 2)[0] for entry in entries], dtype=np.float32),
        "emask_horizontal_radius": np.asarray([_pad_vector(entry.get("Emask_radius", []), 2)[1] for entry in entries], dtype=np.float32),
        "mean_super_depth": np.asarray([_pad_vector(entry.get("mean_depths", []), 2)[0] for entry in entries], dtype=np.float32),
        "mean_deep_depth": np.asarray([_pad_vector(entry.get("mean_depths", []), 2)[1] for entry in entries], dtype=np.float32),
        "super_pos": np.asarray([_pad_vector(entry.get("super_pos", []), 2) for entry in entries], dtype=np.float32),
        "deep_pos": np.asarray([_pad_vector(entry.get("deep_pos", []), 2) for entry in entries], dtype=np.float32),
        "dohough_peak_alphas": np.asarray([_pad_vector(entry.get("alphas", []), max_peaks) for entry in entries], dtype=np.float32),
        "dohough_peak_weights": np.asarray([_pad_vector(entry.get("ws", entry.get("weights", [])), max_peaks) for entry in entries], dtype=np.float32),
        "geofeature_x": np.asarray([_pad_matrix(entry.get("x", []), (max_peaks, 2)) for entry in entries], dtype=np.float32),
        "geofeature_y": np.asarray([_pad_matrix(entry.get("y", []), (max_peaks, 2)) for entry in entries], dtype=np.float32),
        "super_aponeurosis_vector": np.asarray([_pad_vector(entry.get("super_vec_1b", []), len(apox_1b)) for entry in entries], dtype=np.float32),
        "deep_aponeurosis_vector": np.asarray([_pad_vector(entry.get("deep_vec_1b", []), len(apox_1b)) for entry in entries], dtype=np.float32),
        "super_coef": np.asarray([_pad_vector(entry.get("super_coef", []), coef_width) for entry in entries], dtype=np.float32),
        "deep_coef": np.asarray([_pad_vector(entry.get("deep_coef", []), coef_width) for entry in entries], dtype=np.float32),
        "super_coef_linear": np.asarray([_pad_vector(entry.get("super_coef_linear", []), 2) for entry in entries], dtype=np.float32),
        "deep_coef_linear": np.asarray([_pad_vector(entry.get("deep_coef_linear", []), 2) for entry in entries], dtype=np.float32),
        "fas_coef": np.asarray([_pad_vector(entry.get("fas_coef", []), 2) for entry in entries], dtype=np.float32),
        "mm_per_pixel": np.asarray(mm_per_px, dtype=np.float32),
        "frame_step": np.asarray(1, dtype=np.int32),
        "image_height_px": np.asarray(height, dtype=np.int32),
        "image_width_px": np.asarray(width, dtype=np.int32),
        "fps": np.asarray(fps, dtype=np.float32),
    }
    arrays["FL_mm"] = arrays["FL_px"] * np.float32(mm_per_px)
    return arrays


def arrays_to_timtrack_entries(arrays: dict[str, np.ndarray]) -> list[dict]:
    n = len(np.asarray(arrays["frame"]).reshape(-1))
    entries = []
    for idx in range(n):
        entries.append({
            "frame": int(arrays["frame"][idx]),
            "alpha": float(arrays["ANG_deg"][idx]),
            "phi": float(arrays["PEN_deg"][idx]),
            "faslen": float(arrays["FL_px"][idx]),
            "betha": float(arrays["super_apo_angle_deg"][idx]),
            "gamma": float(arrays["deep_apo_angle_deg"][idx]),
            "thickness": float(arrays["muscle_thickness_px"][idx]),
            "apo_x": float(arrays["apo_x"][idx]),
            "super_pos": np.asarray(arrays["super_pos"][idx], dtype=float),
            "deep_pos": np.asarray(arrays["deep_pos"][idx], dtype=float),
            "alphas": np.asarray(arrays["dohough_peak_alphas"][idx], dtype=float),
            "ws": np.asarray(arrays["dohough_peak_weights"][idx], dtype=float),
            "weights": np.asarray(arrays["dohough_peak_weights"][idx], dtype=float),
            "x": np.asarray(arrays["geofeature_x"][idx], dtype=float),
            "y": np.asarray(arrays["geofeature_y"][idx], dtype=float),
            "super_vec_1b": np.asarray(arrays["super_aponeurosis_vector"][idx], dtype=float),
            "deep_vec_1b": np.asarray(arrays["deep_aponeurosis_vector"][idx], dtype=float),
            "super_coef": _trim_nan_tail(arrays["super_coef"][idx]),
            "deep_coef": _trim_nan_tail(arrays["deep_coef"][idx]),
            "super_coef_linear": _trim_nan_tail(arrays["super_coef_linear"][idx]),
            "deep_coef_linear": _trim_nan_tail(arrays["deep_coef_linear"][idx]),
            "fas_coef": _trim_nan_tail(arrays["fas_coef"][idx]),
        })
    return entries


def metric(label: str, reference, estimate, unit: str, target_rmse: float | None = None) -> dict:
    row = metric_row(label, reference, estimate)
    row["unit"] = unit
    row["target_rmse"] = target_rmse
    row["passes"] = bool(target_rmse is None or (np.isfinite(row["rmse"]) and row["rmse"] <= target_rmse))
    return row

In [10]:
if IMAGE_TIMTRACK_NPZ.exists() and not FORCE_REBUILD_TIMTRACK:
    print("Loading cached image-derived TimTrack arrays:", IMAGE_TIMTRACK_NPZ)
    with np.load(IMAGE_TIMTRACK_NPZ, allow_pickle=False) as data:
        image_timtrack = {key: data[key] for key in data.files}
    py_geofeatures = arrays_to_timtrack_entries(image_timtrack)
else:
    print("Generating image-derived TimTrack geofeatures from video frames...")
    start = time.time()
    py_geofeatures = run_timtrack_geofeatures_from_video(
        str(VIDEO_PATH),
        parms,
        limit=mat_n,
        subtraction_mode="matlab_literal",
        keep_debug=False,
        progress_every=250,
    )
    print(f"Image-derived TimTrack runtime: {time.time() - start:.1f}s")
    image_timtrack = entries_to_timtrack_arrays(py_geofeatures)
    np.savez(IMAGE_TIMTRACK_NPZ, **image_timtrack)
    print("Saved:", IMAGE_TIMTRACK_NPZ)

image_alpha = np.asarray(image_timtrack["ANG_deg"], dtype=float).reshape(-1)[:mat_n]
image_peak_alphas = np.asarray(image_timtrack["dohough_peak_alphas"], dtype=float)[:mat_n]
image_peak_weights = np.asarray(image_timtrack["dohough_peak_weights"], dtype=float)[:mat_n]
image_alpha_from_peaks = np.asarray([
    alpha_from_saved_peaks(alphas, weights)
    for alphas, weights in zip(image_peak_alphas, image_peak_weights)
], dtype=float)

print({
    "image_timtrack_frames": len(py_geofeatures),
    "alpha_first5": image_alpha[:5].tolist(),
    "alpha_from_peaks_matches_entry_max_abs": float(np.nanmax(np.abs(image_alpha_from_peaks - image_alpha[:len(image_alpha_from_peaks)]))),
    "cache": str(IMAGE_TIMTRACK_NPZ),
})

Loading cached image-derived TimTrack arrays: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook51_image_derived_timtrack_klt_pipeline_gate/image_derived_timtrack_geofeatures_arrays.npz
{'image_timtrack_frames': 2666, 'alpha_first5': [18.5, 18.5, 19.0, 19.0, 19.0], 'alpha_from_peaks_matches_entry_max_abs': 0.0, 'cache': '/Users/grosbedou/PycharmProjects/NDORMS/results/notebook51_image_derived_timtrack_klt_pipeline_gate/image_derived_timtrack_geofeatures_arrays.npz'}


In [11]:
peak_rows = [
    metric("saved_peak_stream_rebuild", mat_alpha_saved, mat_alpha_rebuilt, "deg", 1e-9),
    metric("image_alpha_entry_vs_saved", mat_alpha_saved, image_alpha, "deg", 1.0),
    metric("image_alpha_peaks_vs_saved", mat_alpha_saved, image_alpha_from_peaks, "deg", 1.0),
    metric("image_top_peak_alpha_vs_saved_top", saved_peak_alphas[:, 0], image_peak_alphas[:, 0], "deg", None),
    metric("image_top_peak_weight_vs_saved_top", saved_peak_weights[:, 0], image_peak_weights[:, 0], "hough weight", None),
]
peak_metrics = pd.DataFrame(peak_rows)
peak_metrics_path = OUT / "image_timtrack_peak_stream_metrics.csv"
peak_metrics.to_csv(peak_metrics_path, index=False)
print("Saved:", peak_metrics_path)
display(peak_metrics.round(6))

worst_alpha = pd.DataFrame({
    "frame": np.arange(mat_n, dtype=int),
    "matlab_saved_alpha_deg": mat_alpha_saved,
    "python_image_alpha_deg": image_alpha,
    "error_deg": image_alpha - mat_alpha_saved,
    "python_top_peak_alpha_deg": image_peak_alphas[:, 0],
    "matlab_top_peak_alpha_deg": saved_peak_alphas[:, 0],
}).assign(abs_error_deg=lambda df: df["error_deg"].abs()).sort_values("abs_error_deg", ascending=False).head(20)
worst_alpha_path = OUT / "worst_image_alpha_frames.csv"
worst_alpha.to_csv(worst_alpha_path, index=False)
print("Saved:", worst_alpha_path)
display(worst_alpha.round(4))

Saved: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook51_image_derived_timtrack_klt_pipeline_gate/image_timtrack_peak_stream_metrics.csv


,comparison,n,bias,mae,rmse,corr,unit,target_rmse,passes
0,saved_peak_stream_rebuild,2666,0.000000,0.000000,0.000000,1.000000,deg,0.0,True
1,image_alpha_entry_vs_saved,2666,-4.345274,4.673481,6.201488,0.568455,deg,1.0,False
2,image_alpha_peaks_vs_saved,2666,-4.345274,4.673481,6.201488,0.568455,deg,1.0,False
3,image_top_peak_alpha_vs_saved_top,2666,-3.944486,5.996999,8.078978,0.189303,deg,NaN,True
4,image_top_peak_weight_vs_saved_top,2666,17.010878,24.356714,29.676646,0.372636,hough weight,NaN,True


Saved: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook51_image_derived_timtrack_klt_pipeline_gate/worst_image_alpha_frames.csv


,frame,matlab_saved_alpha_deg,python_image_alpha_deg,error_deg,python_top_peak_alpha_deg,matlab_top_peak_alpha_deg,abs_error_deg
2533,2533,39.5,19.5,-20.0,10.0,34.0,20.0
1722,1722,33.5,14.0,-19.5,10.0,33.5,19.5
691,691,34.0,15.5,-18.5,13.0,32.0,18.5
955,955,32.0,14.5,-17.5,14.0,33.5,17.5
154,154,33.5,16.0,-17.5,18.5,35.0,17.5
121,121,32.0,15.0,-17.0,28.5,34.0,17.0
136,136,33.5,16.5,-17.0,19.0,35.0,17.0
1725,1725,36.0,19.0,-17.0,19.0,33.0,17.0
137,137,33.5,16.5,-17.0,19.0,35.0,17.0
138,138,33.5,16.5,-17.0,12.0,35.0,17.0


In [12]:
py_timtrack_segments = np.asarray([fascicle_segment_from_geofeature(entry) for entry in py_geofeatures], dtype=float)[:mat_n]
py_super_pos = np.asarray(image_timtrack["super_pos"], dtype=float)[:mat_n]
py_deep_pos = np.asarray(image_timtrack["deep_pos"], dtype=float)[:mat_n]
py_sup = np.column_stack([np.ones(mat_n), py_super_pos[:, 0], np.full(mat_n, float(width)), py_super_pos[:, 1]])
py_deep = np.column_stack([np.ones(mat_n), py_deep_pos[:, 0], np.full(mat_n, float(width)), py_deep_pos[:, 1]])

klt_config = UltraTrackKLTConfig(lk_win_size=win_size)
start = time.time()
py_mask_klt = run_one_step_affine_video(
    VIDEO_PATH,
    py_geofeatures,
    mat_raw_klt,
    super_cut=np.asarray(parms["apo"]["super"]["cut"], dtype=float).reshape(-1),
    config=klt_config,
    limit=mat_n,
    progress_every=500,
)
print(f"Python-geofeature-mask KLT runtime: {time.time() - start:.1f}s")

py_masks_matlab_prior = np.asarray(py_mask_klt["fascicle_segments"], dtype=float)[:mat_n]
py_masks_python_prior = np.full_like(py_timtrack_segments, np.nan, dtype=float)
py_masks_python_prior[0] = py_timtrack_segments[0]
for frame in range(1, min(mat_n, len(py_mask_klt["f_affine_matrices"]))):
    affine = np.asarray(py_mask_klt["f_affine_matrices"][frame], dtype=float)
    if np.all(np.isfinite(affine)):
        py_masks_python_prior[frame] = apply_affine_1b(py_timtrack_segments[frame - 1], affine)

np.savez(
    KLT_NPZ,
    frame=np.arange(mat_n, dtype=np.int32),
    py_masks_matlab_prior_segments=py_masks_matlab_prior.astype(np.float32),
    py_masks_python_timtrack_prior_segments=py_masks_python_prior.astype(np.float32),
    python_timtrack_segments=py_timtrack_segments.astype(np.float32),
    f_points_count=py_mask_klt["f_points_count"],
    f_tracked_count=py_mask_klt["f_tracked_count"],
    f_inlier_count=py_mask_klt["f_inlier_count"],
    f_affine_ok=py_mask_klt["f_affine_ok"],
    f_affine_matrices=py_mask_klt["f_affine_matrices"],
    win_size=np.asarray(win_size, dtype=np.int32),
    mm_per_px=np.asarray(mm_per_px, dtype=np.float32),
)
print("Saved:", KLT_NPZ)
print({
    "affine_rate": float(np.mean(py_mask_klt["f_affine_ok"][1:])),
    "mean_points": float(np.mean(py_mask_klt["f_points_count"][1:])),
    "mean_tracked": float(np.mean(py_mask_klt["f_tracked_count"][1:])),
    "mean_inliers": float(np.mean(py_mask_klt["f_inlier_count"][1:])),
})

one-step KLT processed 501/2666
one-step KLT processed 1001/2666
one-step KLT processed 1501/2666
one-step KLT processed 2001/2666
one-step KLT processed 2501/2666
one-step KLT processed 2666/2666
Python-geofeature-mask KLT runtime: 12.0s
Saved: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook51_image_derived_timtrack_klt_pipeline_gate/python_geofeature_mask_one_step_klt_arrays.npz
{'affine_rate': 1.0, 'mean_points': 299.21013133208254, 'mean_tracked': 299.21013133208254, 'mean_inliers': 299.2003752345216}


In [13]:
valid = np.arange(mat_n) > 0
klt_rows = []
for name, estimate in [
    ("py_masks_matlab_klt_prior", py_masks_matlab_prior),
    ("py_masks_python_timtrack_prior", py_masks_python_prior),
    ("python_timtrack_segment_direct", py_timtrack_segments),
]:
    est = np.asarray(estimate, dtype=float)[:mat_n]
    ref = mat_raw_klt[:len(est)]
    v = valid[:len(est)]
    rows = [
        metric(f"{name}_angle_deg", normalized_angles_deg(ref[v]), normalized_angles_deg(est[v]), "deg", 1.0),
        metric(f"{name}_length_mm", line_lengths_batch(ref[v]) * mm_per_px, line_lengths_batch(est[v]) * mm_per_px, "mm", 2.0),
        metric(f"{name}_x_sup_px", ref[v, 0], est[v, 0], "px", 2.0),
        metric(f"{name}_x_deep_px", ref[v, 2], est[v, 2], "px", 2.0),
    ]
    for row in rows:
        row["variant"] = name
    klt_rows.extend(rows)

klt_metrics = pd.DataFrame(klt_rows)
klt_metrics_path = OUT / "python_geofeature_mask_klt_metrics.csv"
klt_metrics.to_csv(klt_metrics_path, index=False)
print("Saved:", klt_metrics_path)
display(klt_metrics.round(4))

Saved: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook51_image_derived_timtrack_klt_pipeline_gate/python_geofeature_mask_klt_metrics.csv


,comparison,n,bias,mae,rmse,corr,unit,target_rmse,passes,variant
0,py_masks_matlab_klt_prior_angle_deg,2665,0.0009,0.0115,0.0172,1.0000,deg,1.0,True,py_masks_matlab_klt_prior
1,py_masks_matlab_klt_prior_length_mm,2665,-0.0021,0.0197,0.0310,1.0000,mm,2.0,True,py_masks_matlab_klt_prior
2,py_masks_matlab_klt_prior_x_sup_px,2665,-0.0055,0.1112,0.1751,0.9999,px,2.0,True,py_masks_matlab_klt_prior
3,py_masks_matlab_klt_prior_x_deep_px,2665,0.0197,0.1345,0.2111,1.0000,px,2.0,True,py_masks_matlab_klt_prior
4,py_masks_python_timtrack_prior_angle_deg,2665,-4.4395,4.5717,5.9492,0.5904,deg,1.0,False,py_masks_python_timtrack_prior
5,py_masks_python_timtrack_prior_length_mm,2665,6.3967,8.0887,10.6591,0.5447,mm,2.0,False,py_masks_python_timtrack_prior
6,py_masks_python_timtrack_prior_x_sup_px,2665,-30.2299,34.1454,53.5131,0.4164,px,2.0,False,py_masks_python_timtrack_prior
7,py_masks_python_timtrack_prior_x_deep_px,2665,-115.2790,119.3218,146.9182,0.5597,px,2.0,False,py_masks_python_timtrack_prior
8,python_timtrack_segment_direct_angle_deg,2665,-4.4407,4.5727,5.9485,0.5909,deg,1.0,False,python_timtrack_segment_direct
9,python_timtrack_segment_direct_length_mm,2665,6.3979,8.0885,10.6592,0.5447,mm,2.0,False,python_timtrack_segment_direct


In [14]:
kalman_variants = {
    "image_alpha_py_masks_matlab_klt_prior_matlab_apo": {
        "description": "image-derived TimTrack alpha + Python-geofeature KLT masks + MATLAB raw KLT prior + MATLAB aponeuroses",
        "klt": py_masks_matlab_prior,
        "sup": mat_sup,
        "deep": mat_deep,
    },
    "strict_image_timtrack_prior_python_apo": {
        "description": "image-derived TimTrack alpha + Python-geofeature KLT masks + Python TimTrack prior + Python aponeuroses",
        "klt": py_masks_python_prior,
        "sup": py_sup,
        "deep": py_deep,
    },
}

kalman_results = {}
for name, spec in kalman_variants.items():
    kalman_results[name] = run_matlab_2state_kalman(
        np.asarray(spec["klt"], dtype=float)[:mat_n],
        image_alpha[:mat_n],
        np.asarray(spec["sup"], dtype=float)[:mat_n],
        np.asarray(spec["deep"], dtype=float)[:mat_n],
        config=kalman_config,
        mm_per_pixel=mm_per_px,
    )

final_rows = []
for name, result in kalman_results.items():
    for key, unit, target in [("FL_mm", "mm", 2.0), ("PEN_deg", "deg", 1.0), ("ANG_deg", "deg", 1.0)]:
        row = metric(f"{name}_{key}", mat_final[key], result[key], unit, target)
        row["variant"] = name
        row["description"] = kalman_variants[name]["description"]
        final_rows.append(row)

final_metrics = pd.DataFrame(final_rows)
final_metrics = final_metrics[["variant", "description", "comparison", "unit", "n", "bias", "mae", "rmse", "corr", "target_rmse", "passes"]]
final_metrics_path = OUT / "image_timtrack_klt_2state_gate_metrics.csv"
final_metrics.to_csv(final_metrics_path, index=False)
print("Saved:", final_metrics_path)
display(final_metrics.round(4))

variant_summary = []
for name in kalman_variants:
    group = final_metrics[final_metrics["variant"] == name]
    variant_summary.append({
        "variant": name,
        "description": kalman_variants[name]["description"],
        "FL_rmse_mm": float(group.loc[group["comparison"] == f"{name}_FL_mm", "rmse"].iloc[0]),
        "PEN_rmse_deg": float(group.loc[group["comparison"] == f"{name}_PEN_deg", "rmse"].iloc[0]),
        "ANG_rmse_deg": float(group.loc[group["comparison"] == f"{name}_ANG_deg", "rmse"].iloc[0]),
        "passes_final_gate": bool(group["passes"].all()),
    })
summary_df = pd.DataFrame(variant_summary)
summary_table_path = OUT / "image_timtrack_klt_2state_variant_summary.csv"
summary_df.to_csv(summary_table_path, index=False)
print("Saved:", summary_table_path)
display(summary_df.round(4))

Saved: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook51_image_derived_timtrack_klt_pipeline_gate/image_timtrack_klt_2state_gate_metrics.csv


,variant,description,comparison,unit,n,bias,mae,rmse,corr,target_rmse,passes
0,image_alpha_py_masks_matlab_klt_prior_matlab_apo,image-derived TimTrack alpha + Python-geofeatu...,image_alpha_py_masks_matlab_klt_prior_matlab_a...,mm,2666,9.5983,9.5984,10.1987,0.9745,2.0,False
1,image_alpha_py_masks_matlab_klt_prior_matlab_apo,image-derived TimTrack alpha + Python-geofeatu...,image_alpha_py_masks_matlab_klt_prior_matlab_a...,deg,2666,-4.1535,4.1535,4.2607,0.9792,1.0,False
2,image_alpha_py_masks_matlab_klt_prior_matlab_apo,image-derived TimTrack alpha + Python-geofeatu...,image_alpha_py_masks_matlab_klt_prior_matlab_a...,deg,2666,-4.1535,4.1535,4.2607,0.9813,1.0,False
3,strict_image_timtrack_prior_python_apo,image-derived TimTrack alpha + Python-geofeatu...,strict_image_timtrack_prior_python_apo_FL_mm,mm,2666,9.2088,9.4232,12.2422,0.5720,2.0,False
4,strict_image_timtrack_prior_python_apo,image-derived TimTrack alpha + Python-geofeatu...,strict_image_timtrack_prior_python_apo_PEN_deg,deg,2666,-3.9553,4.0100,5.2695,0.6149,1.0,False
5,strict_image_timtrack_prior_python_apo,image-derived TimTrack alpha + Python-geofeatu...,strict_image_timtrack_prior_python_apo_ANG_deg,deg,2666,-4.3129,4.3642,5.6404,0.6258,1.0,False


Saved: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook51_image_derived_timtrack_klt_pipeline_gate/image_timtrack_klt_2state_variant_summary.csv


,variant,description,FL_rmse_mm,PEN_rmse_deg,ANG_rmse_deg,passes_final_gate
0,image_alpha_py_masks_matlab_klt_prior_matlab_apo,image-derived TimTrack alpha + Python-geofeatu...,10.1987,4.2607,4.2607,False
1,strict_image_timtrack_prior_python_apo,image-derived TimTrack alpha + Python-geofeatu...,12.2422,5.2695,5.6404,False


In [15]:
gate_name = "image_alpha_py_masks_matlab_klt_prior_matlab_apo"
strict_name = "strict_image_timtrack_prior_python_apo"
gate = kalman_results[gate_name]
strict = kalman_results[strict_name]

np.savez(
    GATE_FINAL_NPZ,
    frame=np.arange(mat_n, dtype=np.int32),
    FL_mm=gate["FL_mm"],
    PEN_deg=gate["PEN_deg"],
    ANG_deg=gate["ANG_deg"],
    fascicle_segments=gate["fascicle_segments"],
    fascicle_end_segments=gate["fascicle_end_segments"],
    X_plus=gate["X_plus"],
    X_minus=gate["X_minus"],
    timtrack_alpha_deg=image_alpha,
    one_step_klt_segments=py_masks_matlab_prior,
    sup_apo_lines=mat_sup,
    deep_apo_lines=mat_deep,
)
np.savez(
    STRICT_FINAL_NPZ,
    frame=np.arange(mat_n, dtype=np.int32),
    FL_mm=strict["FL_mm"],
    PEN_deg=strict["PEN_deg"],
    ANG_deg=strict["ANG_deg"],
    fascicle_segments=strict["fascicle_segments"],
    fascicle_end_segments=strict["fascicle_end_segments"],
    X_plus=strict["X_plus"],
    X_minus=strict["X_minus"],
    timtrack_alpha_deg=image_alpha,
    one_step_klt_segments=py_masks_python_prior,
    sup_apo_lines=py_sup,
    deep_apo_lines=py_deep,
)
print("Saved:", GATE_FINAL_NPZ)
print("Saved:", STRICT_FINAL_NPZ)

cmd = [
    sys.executable,
    str(ROOT / "scripts" / "compare_ultratimtrack_parity.py"),
    "--matlab-result", str(MATLAB_RESULT),
    "--python-utt", str(GATE_FINAL_NPZ),
    "--python-timtrack", str(IMAGE_TIMTRACK_NPZ),
    "--video", str(VIDEO_PATH),
    "--utt-export", str(MATLAB_EXPORT),
    "--out-csv", str(PARITY_CSV),
]
print("Running compare without --matlab-timtrack-export so TimTrack targets saved full-sequence MATLAB geofeatures.")
print(" ".join(cmd))
run = subprocess.run(cmd, cwd=ROOT, text=True, capture_output=True)
print(run.stdout)
if run.stderr:
    print(run.stderr)
if run.returncode != 0:
    raise RuntimeError(f"compare_ultratimtrack_parity.py failed with exit code {run.returncode}")

parity_df = pd.read_csv(PARITY_CSV)
print("Saved:", PARITY_CSV)
display(parity_df.round(4))

Saved: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook51_image_derived_timtrack_klt_pipeline_gate/image_timtrack_py_masks_matlab_klt_2state_final_arrays.npz
Saved: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook51_image_derived_timtrack_klt_pipeline_gate/strict_image_timtrack_prior_2state_final_arrays.npz
Running compare without --matlab-timtrack-export so TimTrack targets saved full-sequence MATLAB geofeatures.
/Library/Developer/CommandLineTools/usr/bin/python3 /Users/grosbedou/PycharmProjects/NDORMS/scripts/compare_ultratimtrack_parity.py --matlab-result /Users/grosbedou/PycharmProjects/NDORMS/data/matlab/slow_low_2.mat --python-utt /Users/grosbedou/PycharmProjects/NDORMS/results/notebook51_image_derived_timtrack_klt_pipeline_gate/image_timtrack_py_masks_matlab_klt_2state_final_arrays.npz --python-timtrack /Users/grosbedou/PycharmProjects/NDORMS/results/notebook51_image_derived_timtrack_klt_pipeline_gate/image_derived_timtrack_geofeatures_arrays.npz --video /Users/

,comparison,n,bias,mae,rmse,corr
0,final_FL_mm,2666,9.5983,9.5984,10.1987,0.9745
1,final_PEN_deg,2666,-4.1535,4.1535,4.2607,0.9792
2,final_ANG_deg,2666,-4.1535,4.1535,4.2607,0.9813
3,timtrack_alpha_deg,2666,-4.3453,4.6735,6.2015,0.5685
4,timtrack_phi_vs_python_pen_deg,2666,-4.0307,4.4246,5.9071,0.5571
5,timtrack_formula_faslen_px,2666,97.5510,108.8768,143.4468,0.5165
6,timtrack_gamma_deep_apo_deg,2666,-0.3635,0.4082,0.5261,0.4805
7,timtrack_betha_super_apo_deg,2666,-0.3145,0.4304,0.5721,0.1747
8,timtrack_super_pos_y1,2666,-1.1632,3.0265,4.0662,0.0950
9,timtrack_super_pos_y2,2666,2.7088,2.8434,3.5136,0.3488


In [16]:
parity_targets = {
    "final_FL_mm": 2.0,
    "final_PEN_deg": 1.0,
    "final_ANG_deg": 1.0,
    "timtrack_alpha_deg": 1.0,
    "timtrack_phi_vs_python_pen_deg": 1.0,
    "timtrack_formula_faslen_px": 2.0 / mm_per_px,
    "timtrack_gamma_deep_apo_deg": 1.0,
    "timtrack_betha_super_apo_deg": 1.0,
    "timtrack_super_pos_y1": 2.0,
    "timtrack_super_pos_y2": 2.0,
    "timtrack_deep_pos_y1": 2.0,
    "timtrack_deep_pos_y2": 2.0,
}
parity_decision = parity_df.copy()
parity_decision["target_rmse"] = parity_decision["comparison"].map(parity_targets)
parity_decision["passes"] = parity_decision.apply(
    lambda row: bool(pd.isna(row["target_rmse"]) or row["rmse"] <= row["target_rmse"]),
    axis=1,
)
parity_decision_path = OUT / "parity_gate_decision.csv"
parity_decision.to_csv(parity_decision_path, index=False)
print("Saved:", parity_decision_path)
display(parity_decision.round(4))

final_pass = bool(parity_decision[parity_decision["comparison"].isin(["final_FL_mm", "final_PEN_deg", "final_ANG_deg"])] ["passes"].all())
timtrack_pass = bool(parity_decision[parity_decision["comparison"].str.startswith("timtrack_")]["passes"].all())
peak_alpha_pass = bool(peak_metrics.loc[peak_metrics["comparison"] == "image_alpha_entry_vs_saved", "passes"].iloc[0])
klt_angle_pass = bool(klt_metrics.loc[klt_metrics["comparison"] == "py_masks_matlab_klt_prior_angle_deg", "passes"].iloc[0])
klt_length_pass = bool(klt_metrics.loc[klt_metrics["comparison"] == "py_masks_matlab_klt_prior_length_mm", "passes"].iloc[0])
gate_final_pass = bool(summary_df.loc[summary_df["variant"] == gate_name, "passes_final_gate"].iloc[0])
strict_final_pass = bool(summary_df.loc[summary_df["variant"] == strict_name, "passes_final_gate"].iloc[0])

readiness = pd.DataFrame([
    {
        "gate": "image-derived saved full-sequence TimTrack peak stream",
        "status": "passes" if peak_alpha_pass and timtrack_pass else "fails saved full-sequence MATLAB geofeature parity",
        "ready_for_next": bool(peak_alpha_pass and timtrack_pass),
    },
    {
        "gate": "Python-geofeature masks with one-step KLT handoff",
        "status": "angle and length pass" if klt_angle_pass and klt_length_pass else "fails KLT angle or length parity",
        "ready_for_next": bool(klt_angle_pass and klt_length_pass),
    },
    {
        "gate": "image TimTrack + py masks + MATLAB KLT prior + 2-state Kalman",
        "status": "final FL/PEN/ANG pass" if gate_final_pass and final_pass else "final gate fails",
        "ready_for_next": bool(gate_final_pass and final_pass),
    },
    {
        "gate": "strict image TimTrack prior + py apo + 2-state Kalman",
        "status": "final FL/PEN/ANG pass" if strict_final_pass else "diagnostic variant fails",
        "ready_for_next": bool(strict_final_pass),
    },
])
readiness_path = OUT / "readiness.csv"
readiness.to_csv(readiness_path, index=False)
display(readiness)
print("Saved:", readiness_path)

summary = {
    "notebook": "51_image_derived_timtrack_klt_pipeline_gate.ipynb",
    "uses_saved_matlab_geofeatures_for_timtrack_or_klt_masks": False,
    "compares_timtrack_to_saved_full_sequence_fdat_geofeatures": True,
    "image_timtrack_npz": str(IMAGE_TIMTRACK_NPZ),
    "klt_npz": str(KLT_NPZ),
    "gate_final_npz": str(GATE_FINAL_NPZ),
    "strict_final_npz": str(STRICT_FINAL_NPZ),
    "parity_metrics_csv": str(PARITY_CSV),
    "image_alpha_rmse_deg": float(peak_metrics.loc[peak_metrics["comparison"] == "image_alpha_entry_vs_saved", "rmse"].iloc[0]),
    "timtrack_gate_passes": timtrack_pass,
    "klt_angle_rmse_deg": float(klt_metrics.loc[klt_metrics["comparison"] == "py_masks_matlab_klt_prior_angle_deg", "rmse"].iloc[0]),
    "klt_length_rmse_mm": float(klt_metrics.loc[klt_metrics["comparison"] == "py_masks_matlab_klt_prior_length_mm", "rmse"].iloc[0]),
    "gate_final_passes": bool(gate_final_pass and final_pass),
    "strict_final_passes": strict_final_pass,
    "main_gate_description": kalman_variants[gate_name]["description"],
    "strict_variant_description": kalman_variants[strict_name]["description"],
}
summary_path = OUT / "summary.json"
summary_path.write_text(json.dumps(summary, indent=2))
print("Saved:", summary_path)
print(json.dumps(summary, indent=2))

decision = "passes" if summary["gate_final_passes"] else "does not pass yet"
display(Markdown(f"**Decision:** the image-derived TimTrack/KLT/Kalman gate {decision}. See `{PARITY_CSV}` for the authoritative parity rows."))

Saved: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook51_image_derived_timtrack_klt_pipeline_gate/parity_gate_decision.csv


,comparison,n,bias,mae,rmse,corr,target_rmse,passes
0,final_FL_mm,2666,9.5983,9.5984,10.1987,0.9745,2.0000,False
1,final_PEN_deg,2666,-4.1535,4.1535,4.2607,0.9792,1.0000,False
2,final_ANG_deg,2666,-4.1535,4.1535,4.2607,0.9813,1.0000,False
3,timtrack_alpha_deg,2666,-4.3453,4.6735,6.2015,0.5685,1.0000,False
4,timtrack_phi_vs_python_pen_deg,2666,-4.0307,4.4246,5.9071,0.5571,1.0000,False
5,timtrack_formula_faslen_px,2666,97.5510,108.8768,143.4468,0.5165,22.1696,False
6,timtrack_gamma_deep_apo_deg,2666,-0.3635,0.4082,0.5261,0.4805,1.0000,True
7,timtrack_betha_super_apo_deg,2666,-0.3145,0.4304,0.5721,0.1747,1.0000,True
8,timtrack_super_pos_y1,2666,-1.1632,3.0265,4.0662,0.0950,2.0000,False
9,timtrack_super_pos_y2,2666,2.7088,2.8434,3.5136,0.3488,2.0000,False


,gate,status,ready_for_next
0,image-derived saved full-sequence TimTrack pea...,fails saved full-sequence MATLAB geofeature pa...,False
1,Python-geofeature masks with one-step KLT handoff,angle and length pass,True
2,image TimTrack + py masks + MATLAB KLT prior +...,final gate fails,False
3,strict image TimTrack prior + py apo + 2-state...,diagnostic variant fails,False


Saved: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook51_image_derived_timtrack_klt_pipeline_gate/readiness.csv
Saved: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook51_image_derived_timtrack_klt_pipeline_gate/summary.json
{
  "notebook": "51_image_derived_timtrack_klt_pipeline_gate.ipynb",
  "uses_saved_matlab_geofeatures_for_timtrack_or_klt_masks": false,
  "compares_timtrack_to_saved_full_sequence_fdat_geofeatures": true,
  "image_timtrack_npz": "/Users/grosbedou/PycharmProjects/NDORMS/results/notebook51_image_derived_timtrack_klt_pipeline_gate/image_derived_timtrack_geofeatures_arrays.npz",
  "klt_npz": "/Users/grosbedou/PycharmProjects/NDORMS/results/notebook51_image_derived_timtrack_klt_pipeline_gate/python_geofeature_mask_one_step_klt_arrays.npz",
  "gate_final_npz": "/Users/grosbedou/PycharmProjects/NDORMS/results/notebook51_image_derived_timtrack_klt_pipeline_gate/image_timtrack_py_masks_matlab_klt_2state_final_arrays.npz",
  "strict_final_npz": "/Users/grosb

**Decision:** the image-derived TimTrack/KLT/Kalman gate does not pass yet. See `/Users/grosbedou/PycharmProjects/NDORMS/results/notebook51_image_derived_timtrack_klt_pipeline_gate/parity_metrics.csv` for the authoritative parity rows.